# SPLADE Re-Ranking Pipeline on MS MARCO Dev

**Goal**: BM25 top-1000 retrieval → SPLADE re-ranking → evaluate MRR@10, Recall@K

| Step | What |
|------|------|
| 1 | Load MS MARCO dev queries (~6,980) + qrels via `ir_datasets` |
| 2 | BM25 first-stage: top-1000 candidates per query |
| 3 | For each query: encode query + its BM25 candidates (with doc-level caching) |
| 4 | Re-rank via sparse dot-product, update running metrics |
| 5 | Final evaluation summary |

In [ ]:
# Cell 1 — Install dependencies
%pip install ir-datasets pytrec-eval torch transformers omegaconf tqdm

In [ ]:
# Cell 2 — Configuration
# All hyperparameters in one place. Change MAX_QUERIES to a small number for quick testing.

CONFIG = dict(
    model_name       = "naver/splade-cocondenser-ensembledistil",
    collection_path  = "collection.tsv",
    ir_dataset_name  = "msmarco-passage/dev/small",
    batch_size       = 32,            # batch size for SPLADE encoding
    max_length       = 256,           # max token length for passages
    top_k_bm25       = 1000,          # candidates from BM25
    max_queries      = None,          # set to e.g. 100 for fast debugging; None = all ~6980
    log_every        = 100,           # print running metrics every N queries
)

print("Configuration:")
for k, v in CONFIG.items():
    print(f"  {k:20s} = {v}")

In [ ]:
# Cell 3 — Load SPLADE model + tokenizer

import torch
from transformers import AutoTokenizer
from splade.splade.models.transformer_rep import Splade

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

model = Splade(CONFIG["model_name"], agg="max").to(device)
model.eval()

tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])
reverse_voc = {v: k for k, v in tokenizer.vocab.items()}

print(f"Model loaded: {CONFIG['model_name']}")
print(f"Vocab size: {len(tokenizer.vocab)}")

In [ ]:
# Cell 4 — Load MS MARCO dev queries via ir_datasets

import ir_datasets

dataset = ir_datasets.load(CONFIG["ir_dataset_name"])

# Load queries: {query_id: query_text}
queries = {}
for q in dataset.queries_iter():
    queries[q.query_id] = q.text

print(f"Total queries loaded: {len(queries)}")
print(f"Sample: {list(queries.items())[:3]}")

In [ ]:
# Cell 5 — Load qrels into pytrec_eval format
# Format: {query_id: {doc_id: relevance_int}}

qrels = {}
for qrel in dataset.qrels_iter():
    qid = qrel.query_id
    did = qrel.doc_id
    rel = qrel.relevance
    if qid not in qrels:
        qrels[qid] = {}
    qrels[qid][did] = rel

print(f"Queries with relevance judgments: {len(qrels)}")
print(f"Sample qrel: {list(qrels.items())[0]}")

In [ ]:
# Cell 6 — BM25 first-stage retrieval: top-1000
# ir_datasets provides the official MS MARCO BM25 top-1000 dev set via scoreddocs.

from collections import defaultdict
from tqdm import tqdm

bm25_run = defaultdict(dict)  # {qid: {did: score}}

print("Loading BM25 top-1000 scored docs from ir_datasets...")
for sd in tqdm(dataset.scoreddocs_iter(), desc="BM25 scoreddocs"):
    qid = sd.query_id
    # Respect MAX_QUERIES: only load queries we care about
    if CONFIG["max_queries"] and len(bm25_run) >= CONFIG["max_queries"] and qid not in bm25_run:
        continue
    if len(bm25_run[qid]) < CONFIG["top_k_bm25"]:
        bm25_run[qid][sd.doc_id] = sd.score

bm25_run = dict(bm25_run)

print(f"Queries in BM25 run: {len(bm25_run)}")
avg_docs = sum(len(v) for v in bm25_run.values()) / max(1, len(bm25_run))
print(f"Avg docs per query: {avg_docs:.1f}")

# Restrict queries dict to only those in the BM25 run
queries = {qid: text for qid, text in queries.items() if qid in bm25_run}
print(f"Queries to process: {len(queries)}")

In [ ]:
# Cell 7 — Evaluate BM25 baseline

import pytrec_eval

METRICS = ("recall_10", "recall_100", "recall_1000", "recip_rank")

def evaluate_run(run, qrels, metrics=METRICS):
    """Evaluate a run dict against qrels using pytrec_eval."""
    qrels_filtered = {qid: rels for qid, rels in qrels.items() if qid in run}
    evaluator = pytrec_eval.RelevanceEvaluator(qrels_filtered, set(metrics))
    results = evaluator.evaluate(run)
    agg = {}
    for metric in metrics:
        values = [r[metric] for r in results.values() if metric in r]
        agg[metric] = sum(values) / max(1, len(values))
    return agg


def find_relevant_rank(scored_docs, relevant_doc_ids):
    """Find the rank (1-based) of the first relevant document in scored_docs.
    
    Args:
        scored_docs: {doc_id: score} dict
        relevant_doc_ids: set/dict of relevant doc_ids
    
    Returns:
        rank (1-based) of first relevant doc, or None if not found
    """
    sorted_dids = sorted(scored_docs, key=scored_docs.get, reverse=True)
    for rank, did in enumerate(sorted_dids, 1):
        if did in relevant_doc_ids:
            return rank
    return None


bm25_run_eval = {qid: {did: float(s) for did, s in docs.items()} for qid, docs in bm25_run.items()}
bm25_metrics = evaluate_run(bm25_run_eval, qrels)

print("=" * 50)
print("BM25 Baseline Metrics")
print("=" * 50)
for metric, value in bm25_metrics.items():
    print(f"  {metric:20s} = {value:.4f}")

In [ ]:
# Cell 8 — Load full collection

import csv

print(f"Loading collection from {CONFIG['collection_path']}...")

collection = {}  # {doc_id: text}

with open(CONFIG["collection_path"], encoding="utf-8") as f:
    reader = csv.reader(f, delimiter="\t")
    for row in tqdm(reader, desc="Loading collection"):
        doc_id, text = row[0], row[1]
        collection[doc_id] = text

print(f"Collection size: {len(collection):,} passages")
print(f"Sample: {list(collection.items())[0]}")

In [ ]:
# Cell 9 — SPLADE encoding helpers with doc-level cache (SPARSE storage)
#
# Memory optimization: SPLADE vectors are very sparse (~100-200 non-zero values
# out of 30522). Storing as dense float32 = 122 KB/doc → 122 GB for 1M docs.
# Sparse COO storage: ~1.8 KB/doc → ~1.8 GB for 1M docs (~68x reduction).

VOCAB_SIZE = len(tokenizer.vocab)  # 30522
doc_rep_cache = {}  # {doc_id: sparse_coo_tensor}


def _to_sparse(dense_vec):
    """Convert a 1-D dense tensor to a sparse COO tensor for compact storage."""
    return dense_vec.to_sparse_coo().coalesce()


def _sparse_to_dense(sparse_vec):
    """Convert a sparse COO tensor back to dense for computation."""
    return sparse_vec.to_dense()


def encode_query(query_text):
    """Encode a single query with SPLADE. Returns a 1-D dense CPU tensor (vocab_size,)."""
    tokens = tokenizer(
        [query_text],
        return_tensors="pt",
        truncation=True,
        max_length=CONFIG["max_length"],
        padding=True,
    )
    tokens = {k: v.to(device) for k, v in tokens.items()}
    with torch.no_grad():
        q_rep = model(q_kwargs=tokens)["q_rep"].squeeze(0).cpu()
    return q_rep


def encode_docs_cached(doc_ids):
    """Encode docs with SPLADE, caching as SPARSE tensors.
    Returns a dense (len(doc_ids), vocab_size) matrix for dot-product.
    """
    uncached_ids = [did for did in doc_ids if did not in doc_rep_cache]
    if uncached_ids:
        batch_size = CONFIG["batch_size"]
        for i in range(0, len(uncached_ids), batch_size):
            batch_ids = uncached_ids[i : i + batch_size]
            batch_texts = [collection[did] for did in batch_ids]
            tokens = tokenizer(
                batch_texts,
                return_tensors="pt",
                truncation=True,
                max_length=CONFIG["max_length"],
                padding=True,
            )
            tokens = {k: v.to(device) for k, v in tokens.items()}
            with torch.no_grad():
                reps = model(d_kwargs=tokens)["d_rep"]  # (batch, vocab_size) dense
            for j, did in enumerate(batch_ids):
                doc_rep_cache[did] = _to_sparse(reps[j].cpu())  # store SPARSE
    # Densify only the docs needed for this query (~1000 × 122KB = 122MB temporary)
    return torch.stack([_sparse_to_dense(doc_rep_cache[did]) for did in doc_ids])


def rerank_query(qid, query_text, candidate_doc_ids):
    """Encode query + its candidates, compute dot-product scores."""
    q_vec = encode_query(query_text)                        # (vocab_size,) dense
    d_matrix = encode_docs_cached(candidate_doc_ids)        # (n_candidates, vocab_size) dense
    scores = torch.matmul(d_matrix, q_vec).tolist()         # (n_candidates,)
    del d_matrix  # free the temporary dense matrix immediately
    return {did: score for did, score in zip(candidate_doc_ids, scores)}


print(f"Helpers defined. Doc cache size: {len(doc_rep_cache)}")
print(f"Storage: sparse COO (~1.8 KB/doc vs 122 KB/doc dense)")

In [ ]:
# Cell 10 — Streaming re-ranking with running metrics + per-query details
# Process one query at a time: encode → score → record details → update metrics.

splade_run = {}            # full run accumulator: {qid: {did: score}}
query_details = []         # per-query records for metrics.json
log_every = CONFIG["log_every"]
processed = 0

query_ids = list(queries.keys())
pbar = tqdm(query_ids, desc="Re-ranking")

for idx, qid in enumerate(pbar):
    # ── Filter: skip queries with no positive relevance ──
    relevant_docs = {did: rel for did, rel in qrels.get(qid, {}).items() if rel > 0}
    if not relevant_docs:
        continue

    # ── Re-rank ──
    candidate_dids = list(bm25_run[qid].keys())
    splade_run[qid] = rerank_query(qid, queries[qid], candidate_dids)
    processed += 1

    # ── Model's #1 answer ──
    best_did = max(splade_run[qid], key=splade_run[qid].get)
    best_score = splade_run[qid][best_did]
    best_text = collection.get(best_did, "N/A")

    # ── Correct answer(s) ──
    correct_dids = list(relevant_docs.keys())
    correct_text = collection.get(correct_dids[0], "N/A")

    # ── Rank of first relevant doc (denominator in MRR) ──
    rank = find_relevant_rank(splade_run[qid], relevant_docs)
    reciprocal_rank = 1.0 / rank if rank else 0.0

    # ── Record per-query detail ──
    query_details.append({
        "query_idx": processed,
        "qid": qid,
        "query": queries[qid],
        "model_answer": {
            "doc_id": best_did,
            "score": round(best_score, 4),
            "text": best_text[:300],
        },
        "correct_answer": {
            "doc_ids": correct_dids,
            "text": correct_text[:300],
        },
        "relevant_rank": rank,
        "reciprocal_rank": round(reciprocal_rank, 6),
        "is_top1_hit": best_did in relevant_docs,
    })

    # ── Log every N queries ──
    if processed % log_every == 0 or (idx + 1) == len(query_ids):
        running_metrics = evaluate_run(splade_run, qrels)
        mrr = running_metrics.get("recip_rank", 0)
        r10 = running_metrics.get("recall_10", 0)
        r100 = running_metrics.get("recall_100", 0)
        r1000 = running_metrics.get("recall_1000", 0)

        pbar.set_postfix({
            "MRR@10": f"{mrr:.4f}",
            "R@10": f"{r10:.4f}",
            "R@1000": f"{r1000:.4f}",
            "cache": len(doc_rep_cache),
        })

        tqdm.write("")
        tqdm.write(f"  ── Query #{processed} (idx={idx+1}/{len(query_ids)}, qid={qid}) ──")
        tqdm.write(f"  Question : {queries[qid]}")
        tqdm.write(f"  Model #1 : [did={best_did}, score={best_score:.2f}] {best_text[:150]}...")
        tqdm.write(f"  Correct  : [did={correct_dids[0]}" +
                   (f" +{len(correct_dids)-1} more" if len(correct_dids) > 1 else "") +
                   f"] {correct_text[:150]}...")
        tqdm.write(f"  Rank of correct: {rank}  (1/rank = {reciprocal_rank:.4f})")
        tqdm.write(f"  Top-1 hit: {'✓ YES' if best_did in relevant_docs else '✗ NO'}")
        tqdm.write(
            f"  Metrics  : MRR@10={mrr:.4f}  R@10={r10:.4f}  "
            f"R@100={r100:.4f}  R@1000={r1000:.4f}  "
            f"doc_cache={len(doc_rep_cache):,}"
        )

print(f"\nDone. Processed {processed} queries (skipped {len(query_ids) - processed} with empty qrels).")
print(f"Doc cache: {len(doc_rep_cache):,} passages.")
print(f"Query details collected: {len(query_details)}")

In [ ]:
# Cell 11 — Final SPLADE metrics + save metrics.json

import json

splade_metrics = evaluate_run(splade_run, qrels)

# Compute MRR@10 from per-query details: 1/rank if rank <= 10, else 0
mrr_at_10_values = []
for qd in query_details:
    rank = qd['relevant_rank']
    mrr_at_10_values.append(1.0 / rank if rank is not None and rank <= 10 else 0.0)
mrr_at_10 = sum(mrr_at_10_values) / max(1, len(mrr_at_10_values))
splade_metrics['mrr@10'] = mrr_at_10

print("=" * 50)
print("SPLADE Re-Ranking Metrics (final)")
print("=" * 50)
for metric, value in splade_metrics.items():
    print(f"  {metric:20s} = {value:.4f}")

# Build and save full metrics JSON
metrics_output = {
    "bm25": bm25_metrics,
    "splade": splade_metrics,
    "delta": {m: round(splade_metrics.get(m, 0) - bm25_metrics[m], 6) for m in bm25_metrics},
    "config": {
        "model_name": CONFIG["model_name"],
        "max_queries": CONFIG["max_queries"],
        "batch_size": CONFIG["batch_size"],
        "top_k_bm25": CONFIG["top_k_bm25"],
        "queries_processed": processed,
        "doc_cache_size": len(doc_rep_cache),
    },
    "queries": query_details,
}

with open("metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics_output, f, indent=2, ensure_ascii=False)

print(f"\nMetrics + per-query details saved to metrics.json ({len(query_details)} queries)")

In [ ]:
# Cell 12 — Compare BM25 vs SPLADE

from splade_utils import sparse_to_bow, pretty_bow

print("\n" + "=" * 60)
print(f"{'Metric':<22} {'BM25':>10} {'SPLADE':>10} {'Delta':>10}")
print("-" * 60)
for metric in METRICS:
    bm25_val = bm25_metrics[metric]
    splade_val = splade_metrics.get(metric, 0)
    delta = splade_val - bm25_val
    print(f"  {metric:<20} {bm25_val:>10.4f} {splade_val:>10.4f} {delta:>+10.4f}")
print(f"  {'mrr@10':<20} {'N/A':>10} {splade_metrics['mrr@10']:>10.4f} {'':>10}")
print("=" * 60)

# Show a sample query
sample_qid = list(splade_run.keys())[0]
print(f"\nSample query (qid={sample_qid}): {queries[sample_qid]}")
sample_q_rep = encode_query(queries[sample_qid])
q_bow = sparse_to_bow(sample_q_rep, reverse_voc, top_k=30)
print(pretty_bow(q_bow, max_terms=15))

# Show top-5 re-ranked docs for this query
sorted_docs = sorted(splade_run[sample_qid].items(), key=lambda x: x[1], reverse=True)[:5]
print(f"\nTop-5 SPLADE re-ranked docs for qid={sample_qid}:")
for rank, (did, score) in enumerate(sorted_docs, 1):
    relevant = "✓ RELEVANT" if did in qrels.get(sample_qid, {}) else ""
    print(f"  #{rank}  doc_id={did}  score={score:.4f}  {relevant}")
    print(f"       {collection[did][:120]}...")